# DriftWise: gate ablation
Does the careful sequential gate beat a simple 'switch if it looks better' rule?

**How to run:**
1. Run cell 1 (install). Then **Runtime > Restart session**.
2. Do NOT run cell 1 again. Run cell 2 (allow Google Drive access when asked).
3. Wait about 5 to 8 minutes and paste the printed tables back into the chat.

Needs `ega_pipeline.py` and `ega_gate.py` in `MyDrive/DriftWise/src/`.

## 1. Install (run once, then Runtime > Restart session)

In [ ]:
!pip install river "numpy<2.3" -q

## 2. Ablation (Part A: pattern drift, Part B: no-drift streams)

In [ ]:
import sys
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
from scipy import stats
sys.path.insert(0, "/content/drive/MyDrive/DriftWise/src")
import ega_pipeline as ep                       # all our functions, restored by one import
seq_gate = ep.sequential_gate                   # the real sequential gate (kept so we can restore it)

def bal(y, p):
    r = [float((p[y == c] == c).mean()) for c in (0, 1)]
    return float(np.mean(r)), float(min(r))

def point_gate(y, p_new, p_old, look=None, alpha_total=None, guard=False, min_n=200, min_per_class=30):
    """Simple 'follow the leader' switch: accept if the candidate's balanced accuracy is higher on the
    validation block. No confidence interval. Same minimum-evidence floor as the real gate."""
    y, p_new, p_old = map(np.asarray, (y, p_new, p_old))
    n1 = int(y.sum()); n0 = len(y) - n1
    if len(y) < min_n or n1 < min_per_class or n0 < min_per_class:
        return {"decision": "DEFER", "est_balacc": 0.0}
    b_new, w_new = bal(y, p_new); b_old, w_old = bal(y, p_old)
    ok = b_new > b_old
    if guard:
        ok = ok and (w_new - w_old > -0.03)      # worst-class recall may not drop by more than 0.03
    return {"decision": "ACCEPT" if ok else "REJECT", "est_balacc": b_new - b_old}

def point_gate_guard(*a, **k):
    return point_gate(*a, guard=True, **k)

GATES = {"sequential gate (ours)": seq_gate,
         "point estimate": point_gate,
         "point estimate + class guard": point_gate_guard}

# ---------------- Part A: pattern drift, 20 fresh seeds ----------------
def run_with_gate(gate, X, y):
    ep.sequential_gate = gate                    # run_ega_v2 looks this name up when it runs
    try:
        ep.reset_cost()
        r = ep.run_ega_v2(X, y, lag=0, patience=3)
        r["models_trained"] = ep.COST["fits"]
    finally:
        ep.sequential_gate = seq_gate
    return r

rows = []
for seed in range(10, 30):
    X, y = ep.make_stream(seed, "pattern_drift")
    for name, g in GATES.items():
        r = run_with_gate(g, X, y)
        rows.append({"gate": name, "seed": seed, **r})
A = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 20)
print("PART A: pattern drift, 20 fresh seeds (mean)")
display(A.groupby("gate")[["balanced_acc", "worst_class_recall", "n_updates", "models_trained"]].mean().round(3))

ref = A[A.gate == "sequential gate (ours)"].set_index("seed")
print("Sequential gate minus each simple rule, paired by seed (mean [95% CI]):")
for name in ["point estimate", "point estimate + class guard"]:
    oth = A[A.gate == name].set_index("seed")
    for m in ["balanced_acc", "worst_class_recall"]:
        d = (ref[m] - oth[m]).dropna()
        sd = d.std(ddof=1)
        h = stats.t.ppf(0.975, len(d) - 1) * sd / np.sqrt(len(d)) if sd > 0 else 0.0
        print(f"  vs {name:30s} {m:19s} {d.mean():+.3f} [{d.mean()-h:+.3f}, {d.mean()+h:+.3f}]")

# ---------------- Part B: streams with NO drift; do the gates accept truly worse models? ----------------
def make_no_drift_stream(seed, n=6000):
    r = np.random.default_rng(seed)
    y = (r.random(n) < 0.05).astype(int)
    X = r.normal(size=(n, 2))
    X[:, 0] += 2.0 * y
    return X, y

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return f"[{c - h:.3f}, {c + h:.3f}]"

val_size, gap, train_size = 1000, 50, 1000
checkpoints = list(range(4500, 6000, 200))
n_streams = 200
rec = []
for seed in range(n_streams):
    X, y = make_no_drift_stream(seed)
    Xt, yt = make_no_drift_stream(seed + 100000, n=20000)       # big fresh test set = the 'truth'
    old = ep.fit_model(X[:3000], y[:3000])
    b_old = bal(yt, ep.predict(old, Xt))[0]
    for look, t in enumerate(checkpoints, start=1):
        te = t + 1 - val_size - gap
        cand = ep.fit_model(X[te - train_size:te], y[te - train_size:te])
        Xv, yv = X[t + 1 - val_size:t + 1], y[t + 1 - val_size:t + 1]
        p_old, p_new = ep.predict(old, Xv), ep.predict(cand, Xv)
        true_diff = bal(yt, ep.predict(cand, Xt))[0] - b_old     # real change in balanced accuracy
        for name, g in GATES.items():
            acc = g(yv, p_new, p_old, look=look)["decision"] == "ACCEPT"
            rec.append({"gate": name, "seed": seed, "accept": acc, "true_diff": true_diff})
B = pd.DataFrame(rec)
n_checks = len(B[B.gate == "sequential gate (ours)"])

out = []
for name in GATES:
    s = B[B.gate == name]; a = s[s.accept]
    harmful = a[a.true_diff < -0.01]
    out.append({"gate": name, "accepts per 1000 checks": round(1000 * len(a) / n_checks, 1),
                "harmful accepts per 1000 checks": round(1000 * len(harmful) / n_checks, 1),
                "share of accepts that are harmful": round(len(harmful) / len(a), 3) if len(a) else None,
                "mean true change of accepted": round(a.true_diff.mean(), 4) if len(a) else None,
                "streams with a harmful accept": f"{harmful.seed.nunique()}/{n_streams} {wilson(harmful.seed.nunique(), n_streams)}"})
allc = B[B.gate == "sequential gate (ours)"]
print("\nPART B:", n_streams, "streams with no drift,", n_checks // n_streams, "gate checks each")
print("Of ALL candidates (accept-everything baseline): truly worse by >0.01:",
      round((allc.true_diff < -0.01).mean(), 3), "| truly better by >0.01:", round((allc.true_diff > 0.01).mean(), 3))
display(pd.DataFrame(out))

import os
os.makedirs("/content/drive/MyDrive/DriftWise/results", exist_ok=True)
A.to_csv("/content/drive/MyDrive/DriftWise/results/ablation_gate_pattern_drift.csv", index=False)
B.to_csv("/content/drive/MyDrive/DriftWise/results/ablation_gate_no_drift.csv", index=False)
print("Saved: ablation_gate_pattern_drift.csv and ablation_gate_no_drift.csv")